<a href="https://colab.research.google.com/github/duruamobi/AAI2026/blob/main/Customer_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import json
import re

def harbortech_support(customer_message: str) -> str:
    """
    HarborTech Home Devices support classifier + responder.

    Process:
      1) Choose intent: {exchange, tracking, troubleshooting}
      2) If required info missing -> ask ONE concise clarification question
      3) Else -> two-sentence response with a clear next step

    Returns JSON string with: intent, clarify, response
    """
    msg = customer_message.strip()
    msg_lower = msg.lower()

    # -----------------------
    # 1) Intent classification
    # -----------------------
    exchange_kw = ["exchange", "swap", "replace", "replacement", "wrong item", "different size", "different model"]
    tracking_kw = ["tracking", "track", "where is", "where's", "delivery", "delivered", "shipment", "shipping", "arrive"]
    troubleshoot_kw = ["not working", "doesn't work", "doesnt work", "broken", "issue", "error", "problem",
                       "won't", "wont", "setup", "pair", "connect", "reset", "troubleshoot"]

    # Simple scoring
    scores = {"exchange": 0, "tracking": 0, "troubleshooting": 0}
    for w in exchange_kw:
        if w in msg_lower:
            scores["exchange"] += 1
    for w in tracking_kw:
        if w in msg_lower:
            scores["tracking"] += 1
    for w in troubleshoot_kw:
        if w in msg_lower:
            scores["troubleshooting"] += 1

    # Pick highest; default to troubleshooting if tie/none
    intent = max(scores, key=scores.get)
    if scores[intent] == 0:
        intent = "troubleshooting"

    # ------------------------------------
    # Helpers: detect required information
    # ------------------------------------
    def extract_order_number(text: str):
        # Common formats: "order #A7X92", "Order ID: 12345", "#BG4521"
        m = re.search(r'(?:order\s*(?:id|number)?\s*[:#]?\s*|#)\s*([A-Z0-9\-]{5,})', text, re.IGNORECASE)
        return m.group(1) if m else None

    def extract_device_model(text: str):
        # Heuristic: "model X123", "HT-200", "HarborTech X100", etc.
        m = re.search(r'(?:model\s*[:#]?\s*)([A-Z0-9\-]{3,})', text, re.IGNORECASE)
        if m:
            return m.group(1)
        # Fallback: look for something like "HT-123" / "HT123"
        m2 = re.search(r'\b(HT[- ]?\d{2,4}[A-Z]?)\b', text, re.IGNORECASE)
        return m2.group(1) if m2 else None

    def has_symptom(text_lower: str) -> bool:
        symptom_markers = ["won't", "wont", "doesn't", "doesnt", "not working", "broken", "error", "issue", "problem", "stuck", "overheating"]
        return any(s in text_lower for s in symptom_markers) and len(text_lower.split()) >= 5

    order_number = extract_order_number(msg)
    device_model = extract_device_model(msg)
    symptom_present = has_symptom(msg_lower)

    # -----------------------------
    # 2) Clarification if missing
    # -----------------------------
    clarify = "NONE"
    response = ""

    if intent in ("exchange", "tracking"):
        # Both require order number
        if not order_number:
            clarify = "What’s your order number?"
            response = "PENDING_INFO"
        else:
            if intent == "exchange":
                response = (
                    "Thanks—I've got your request and can help set up an exchange for that order. "
                    "Next step: please confirm the item you want instead, and I’ll send exchange instructions."
                )
            else:  # tracking
                response = (
                    "Thanks—I can look up the latest shipping status for your order. "
                    "Next step: I’m checking the tracking details now and will share the most recent update."
                )
    else:
        # troubleshooting requires device model OR brief symptom description
        if not device_model and not symptom_present:
            clarify = "Which device model is it, and what’s the main issue you’re seeing?"
            response = "PENDING_INFO"
        else:
            # Use whatever we have to personalize slightly
            model_text = f" for model {device_model}" if device_model else ""
            response = (
                f"Thanks for the details—let’s troubleshoot this{model_text} and get it working again. "
                "Next step: please power-cycle the device (unplug for 30 seconds, then reconnect) and tell me what happens."
            )

    return json.dumps(
        {"intent": intent, "clarify": clarify, "response": response},
        indent=2
    )


# -----------------------------
# Demo examples (Colab-friendly)
# -----------------------------
examples = [
    "I want to exchange my HarborTech humidifier. Order #A7X92",
    "Where is my shipment? I ordered last week.",
    "Tracking update please for Order ID: BG4521",
    "My HarborTech device won't turn on and shows an error code.",
    "Need help, it's not working."
]

for i, ex in enumerate(examples, 1):
    print("=" * 70)
    print(f"DEMO {i}: {ex}")
    print(harbortech_support(ex))
    print()

DEMO 1: I want to exchange my HarborTech humidifier. Order #A7X92
{
  "intent": "exchange",
  "clarify": "NONE",
  "response": "Thanks\u2014I've got your request and can help set up an exchange for that order. Next step: please confirm the item you want instead, and I\u2019ll send exchange instructions."
}

DEMO 2: Where is my shipment? I ordered last week.
{
  "intent": "tracking",
  "clarify": "What\u2019s your order number?",
  "response": "PENDING_INFO"
}

DEMO 3: Tracking update please for Order ID: BG4521
{
  "intent": "tracking",
  "clarify": "NONE",
  "response": "Thanks\u2014I can look up the latest shipping status for your order. Next step: I\u2019m checking the tracking details now and will share the most recent update."
}

DEMO 4: My HarborTech device won't turn on and shows an error code.
{
  "intent": "troubleshooting",
  "clarify": "NONE",
  "response": "Thanks for the details\u2014let\u2019s troubleshoot this and get it working again. Next step: please power-cycle the d